In [9]:
import torch
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler
from diffusers.utils import load_image
from PIL import Image
import PIL
import cv2
import numpy as np
import os 
import matplotlib.pyplot as plt
import random

In [ ]:
def change(content_image_url,style_image_url):

    content_image = cv2.imread(content_image_url)
    style_image = PIL.Image.open(style_image_url)


    canny_image = np.array(content_image)
    canny_image = cv2.Canny(canny_image, 100, 200)
    canny_image = canny_image[:, :, None]
    canny_image = np.concatenate([canny_image, canny_image, canny_image], axis=2)

    canny_image = Image.fromarray(canny_image)

    base_model_path = "runwayml/stable-diffusion-v1-5"
    controlnet_path = "lllyasviel/sd-controlnet-canny"


    controlnet = ControlNetModel.from_pretrained(controlnet_path, torch_dtype=torch.float16)
    pipe = StableDiffusionControlNetPipeline.from_pretrained(
        base_model_path,
        controlnet=controlnet,
        torch_dtype=torch.float16
    ).to("cuda")


    # Load the IP-Adapter weights
    pipe.load_ip_adapter("h94/IP-Adapter", subfolder="models", weight_name="ip-adapter_sd15.bin")

    generator = torch.Generator().manual_seed(42)

    stylized_image = pipe(
        prompt="Image style transfer. Use [Content Image] as the base. Isolate the background region and recolor and re-texture it to match the aesthetic, lighting, and color gradient of the background in [Style Image]. The foreground subjects and objects must remain completely unchanged in form, placement, and size. Focus only on environmental style adaptation.",
        negative_prompt="monochrome, lowres, bad anatomy, worst quality, low quality",
        image=canny_image, # ControlNet input
        ip_adapter_image=style_image, # IP-Adapter input
        num_inference_steps=50,
        generator=generator,
    ).images[0]


    return stylized_image

In [3]:

syn_address = "D:\phd\\2 term 2\Deep Learning\\449.jpg"
real_address = "1\M24_Chaffee_named_'Rebel',_hoodno_USA_30402162_pic3.jpg"
save_address = ""

In [5]:
import cv2
import numpy as np
from matplotlib import pyplot as plt


In [22]:
canny_image = np.array(image)
canny_image = cv2.Canny(canny_image, 100, 200)
canny_image = canny_image[:, :, None]
canny_image = np.concatenate([canny_image, canny_image, canny_image], axis=2)
canny_image = Image.fromarray(canny_image)


In [25]:
x = np.asanyarray(canny_image)

In [31]:
content_address = os.path.join(syn_address)

style_address = os.path.join(real_address)

# create new image 
new_image = change(content_address,style_address)

new_image.save("test2.png")

100%|██████████| 50/50 [01:22<00:00,  1.64s/it]


different edge detection

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def apply_all_edge_detection(image_path):
    # Read and preprocess image
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Convert to RGB for matplotlib
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    
    # Apply Gaussian blur to reduce noise
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    
    # 1. Sobel Edge Detection
    sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)
    sobel_magnitude = np.sqrt(sobel_x**2 + sobel_y**2)
    sobel_magnitude = np.uint8(sobel_magnitude / sobel_magnitude.max() * 255)
    
    # 2. Scharr Edge Detection (more accurate than Sobel)
    scharr_x = cv2.Scharr(blurred, cv2.CV_64F, 1, 0)
    scharr_y = cv2.Scharr(blurred, cv2.CV_64F, 0, 1)
    scharr_magnitude = np.sqrt(scharr_x**2 + scharr_y**2)
    scharr_magnitude = np.uint8(scharr_magnitude / scharr_magnitude.max() * 255)
    
    # 3. Laplacian Edge Detection
    laplacian = cv2.Laplacian(blurred, cv2.CV_64F)
    laplacian_abs = np.uint8(np.absolute(laplacian))
    
    # 4. Canny Edge Detection (with different thresholds)
    canny_low = cv2.Canny(blurred, 50, 150)
    canny_medium = cv2.Canny(blurred, 100, 200)
    canny_high = cv2.Canny(blurred, 150, 250)
    
    # 5. Prewitt Edge Detection (custom implementation)
    def prewitt_edge_detection(img):
        kernel_x = np.array([[-1, 0, 1],
                            [-1, 0, 1],
                            [-1, 0, 1]])
        kernel_y = np.array([[-1, -1, -1],
                            [0, 0, 0],
                            [1, 1, 1]])
        prewitt_x = cv2.filter2D(img, cv2.CV_64F, kernel_x)
        prewitt_y = cv2.filter2D(img, cv2.CV_64F, kernel_y)
        prewitt_magnitude = np.sqrt(prewitt_x**2 + prewitt_y**2)
        return np.uint8(prewitt_magnitude / prewitt_magnitude.max() * 255)
    
    prewitt = prewitt_edge_detection(blurred)
    
    # 6. Roberts Cross Edge Detection (custom implementation)
    def roberts_edge_detection(img):
        kernel_x = np.array([[1, 0],
                            [0, -1]])
        kernel_y = np.array([[0, 1],
                            [-1, 0]])
        roberts_x = cv2.filter2D(img, cv2.CV_64F, kernel_x)
        roberts_y = cv2.filter2D(img, cv2.CV_64F, kernel_y)
        roberts_magnitude = np.sqrt(roberts_x**2 + roberts_y**2)
        return np.uint8(roberts_magnitude / roberts_magnitude.max() * 255)
    
    roberts = roberts_edge_detection(blurred)
    
    # 7. Zero Cross Edge Detection (using Laplacian of Gaussian)
    def zero_cross_edge_detection(img):
        # Apply Laplacian of Gaussian
        log = cv2.GaussianBlur(img, (5, 5), 0)
        log = cv2.Laplacian(log, cv2.CV_64F)
        
        # Simple zero-crossing detection
        zero_cross = np.zeros_like(log, dtype=np.uint8)
        for i in range(1, log.shape[0]-1):
            for j in range(1, log.shape[1]-1):
                if (log[i, j] == 0 or 
                    (log[i, j] > 0 and log[i, j+1] < 0) or
                    (log[i, j] > 0 and log[i, j-1] < 0) or
                    (log[i, j] > 0 and log[i+1, j] < 0) or
                    (log[i, j] > 0 and log[i-1, j] < 0)):
                    zero_cross[i, j] = 255
        return zero_cross
    
    zero_cross = zero_cross_edge_detection(blurred)
    
    return {
        'original': image,
        'gray': gray,
        'sobel': sobel_magnitude,
        'scharr': scharr_magnitude,
        'laplacian': laplacian_abs,
        'canny_low': canny_low,
        'canny_medium': canny_medium,
        'canny_high': canny_high,
        'prewitt': prewitt,
        'roberts': roberts,
        'zero_cross': zero_cross
    }

# Apply all edge detection methods
results = apply_all_edge_detection(syn_address)  # Replace with your image path

# Create the plot
plt.figure(figsize=(20, 15))
plt.suptitle('Comparison of OpenCV Edge Detection Methods', fontsize=16, fontweight='bold')

# Plot all results
methods = [
    ('Original Image', results['original'], 'none'),
    ('Grayscale', results['gray'], 'gray'),
    ('Sobel Edge Detection', results['sobel'], 'gray'),
    ('Scharr Edge Detection', results['scharr'], 'gray'),
    ('Laplacian Edge Detection', results['laplacian'], 'gray'),
    ('Canny (Low Threshold)', results['canny_low'], 'gray'),
    ('Canny (Medium Threshold)', results['canny_medium'], 'gray'),
    ('Canny (High Threshold)', results['canny_high'], 'gray'),
    ('Prewitt Edge Detection', results['prewitt'], 'gray'),
    ('Roberts Cross', results['roberts'], 'gray'),
    ('Zero Cross Detection', results['zero_cross'], 'gray')
]

for i, (title, img, cmap) in enumerate(methods, 1):
    plt.subplot(4, 3, i)
    if cmap == 'none':
        plt.imshow(img)
    else:
        plt.imshow(img, cmap=cmap)
    plt.title(title, fontsize=10, fontweight='bold')
    plt.axis('off')

plt.tight_layout()
plt.subplots_adjust(top=0.94)
plt.show()

# Additional: Save the comparison image
fig, axes = plt.subplots(4, 3, figsize=(20, 15))
axes = axes.ravel()

for i, (title, img, cmap) in enumerate(methods):
    if cmap == 'none':
        axes[i].imshow(img)
    else:
        axes[i].imshow(img, cmap=cmap)
    axes[i].set_title(title, fontsize=10, fontweight='bold')
    axes[i].axis('off')

plt.tight_layout()
plt.subplots_adjust(top=0.94)
plt.savefig('edge_detection_comparison.png', dpi=300, bbox_inches='tight')
plt.close()

print("Comparison image saved as 'edge_detection_comparison.png'")